## Subset Selection

- **Real-World Problem:** Imagine you’re working at a real estate company.
  
- **Goal:** To Predict house price
- **Data you get:**
  
|**Feature**      |	**Meaning**             |      
|--------------|-----------------------------
|size	       | house size             
|rooms	       |  number of rooms       
|age	       |      age of house      |
|random_id     | 	system ID           |
|noise_feature |	unclear metric      |


- **Problem: You don’t know which features truly matter.**
  
- Should you use all?
- Or select only useful ones?
- **Idea of Subset Selection:** Try features step by step and keep only those that improve prediction.

## Implementation (Forward Selection)

### Step 1: Create synthetic data to simulate real world problem


In [24]:
import pandas as pd
import numpy as np

np.random.seed(42)
# Meaningful features
size = np.random.normal(1500, 300, 100)
rooms = np.random.randint(1, 6, 100)
age = np.random.randint(1, 30, 100)

# less useful features / noise

random_id = np.random.randint(1000, 5000, 100)
noise_feature = np.random.randn(100)

# target (hidden truth)
price = 200*size + 10000*rooms -500*age + np.random.randn(100)*10000

# dataset
df = pd.DataFrame({
    "size": size,
    "rooms": rooms,
    "age": age,
    "random_id": random_id,
    "noise_feature": noise_feature,
    "price": price
})

df.head()

,size,rooms,age,random_id,noise_feature,price
0,1649.014246,1,22,2500,-0.319355,318609.434508
1,1458.520710,5,3,2705,-0.204617,348523.723795
2,1694.306561,1,16,3806,0.271062,333617.974333
3,1956.908957,3,29,1537,2.244807,388124.004999
4,1429.753988,2,9,2890,-0.787602,295708.783029


## Step 2: Evaluation function

This function checks how good a set of features is.

What it does:
- it Takes selected features
- Trains a model
- Predicts price
- Returns error (MSE)

>Lower error = better features

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def evaluate(features):
    X = df[features]
    y = df["price"]
    
    model = LinearRegression()
    model.fit(X, y)
    preds = model.predict(X)
    
    return mean_squared_error(y, preds)

## Step 3: Forward Feature Selection
We build the best feature set step-by-step.
  



#### 🎯 **Objective**

Select a subset of features that improves model performance, while avoiding irrelevant or noisy features.

---

#### 💡 **Core Idea**

* Start with no features
* Iteratively add one feature at a time
* At each step:

  * Evaluate all remaining features
  * Select the feature that reduces error the most
* Stop when additional features provide insignificant improvement

---

#### 🧩 Problem Setup

Given features:

* `size` → meaningful
* `rooms` → meaningful
* `age` → moderately useful
* `random_id` → irrelevant
* `noise_feature` → random noise

Goal:

* Identify which features actually contribute to predicting the target (e.g., price)

---

#### 🔁 Algorithm Workflow

##### 1. Initialization

* `selected = []` → no features selected initially
* `remaining = [all features]` → features available for selection
* `prev_score = ∞` → initial baseline (worst possible error)

---

##### 2. Iterative Selection Process

Repeat until stopping condition is met:

###### a. Evaluate Each Candidate Feature

* For every feature `f` in `remaining`:

  * Temporarily form:

    ```
    current_features = selected + [f]
    ```
  * Compute model error:

    ```
    score = evaluate(current_features)
    ```
  * Store `(score, f)` for comparison

---

###### b. Select Best Feature

* Sort all evaluated features by score
* Choose the feature with **lowest error**

  ```
  best_feature = feature with minimum score
  best_score = corresponding error
  ```

---

###### c. Check Stopping Condition

* Compute improvement:

  ```
  improvement = prev_score - best_score
  ```
* If improvement is too small:

  * Stop the algorithm (`break`)

---

###### d. Update State (if continuing)

* Add feature to selected set:

  ```
  selected.append(best_feature)
  ```
* Remove from remaining:

  ```
  remaining.remove(best_feature)
  ```
* Update baseline score:

  ```
  prev_score = best_score
  ```

---

#### 🔄 Loop Control Intuition

* The `while` loop:

  * Continues as long as there are features left **and improvement exists**
* The `break` statement:

  * Immediately stops the loop when no meaningful improvement is observed
* Updates (`selected`, `remaining`, `prev_score`) occur **only if the loop continues**

---

#### 🧠 Key Insights

* Greedy approach:

  * Makes the best choice at each step (locally optimal)
* Builds model **incrementally**
* Helps avoid:

  * Overfitting
  * Inclusion of irrelevant features
* Performance depends heavily on:

  * The `evaluate()` function

---

#### ⚠️ Limitations

* May miss optimal feature combinations (not globally optimal)
* Computationally expensive for large feature sets
* Sensitive to evaluation method (training vs validation error)

---

#### 🔥 Summary

* Start with no features
* Add the best feature at each step
* Stop when improvement becomes negligible

> Forward Selection = Incremental, greedy feature selection based on performance improvement

#re at each step
* Stop when improvement becomes negligible

> Forward Selection = Incremental, greedy feature selection based on performance improvement


In [83]:
remaining = ['size', 'rooms', 'age', 'random_id', 'noise_feature']
selected = []
prev_score = float('inf')

while remaining:
    scores = []
    
    # try each remaining feature
    for f in remaining:
        current_features = selected + [f]
        score = evaluate(current_features)
        scores.append((score, f))
    
    # pick best feature (lowest error)
    scores.sort()
    best_score, best_feature = scores[0]
    
    # stop if no improvement
    if prev_score - best_score < 1000:
        break
    
    selected.append(best_feature)
    remaining.remove(best_feature)
    prev_score = best_score
    
    print(f"Added: {best_feature}, MSE: {best_score:.2f}")

print("\nFinal selected features:", selected)

Added: size, MSE: 285862809.36
Added: rooms, MSE: 133115169.05
Added: age, MSE: 110073785.45
Added: random_id, MSE: 108305241.73
Added: noise_feature, MSE: 107468770.44

Final selected features: ['size', 'rooms', 'age', 'random_id', 'noise_feature']


## Important Note on Evaluation Strategy (Training Error vs Generalization)

### 🧠 What this implementation does

In the current setup, the evaluation function is defined as:

```python
model.fit(X, y)
preds = model.predict(X)
return mean_squared_error(y, preds)
```

* The model is trained on the dataset `(X, y)`
* Predictions are made on the **same dataset**
* Error (MSE) is computed on the **same data used for train#ing**

---

## 🚨 Problem: Training Error is Misleading

This approach measures **training error**, not real performance.

### Key issue:

* As more features are added, the model becomes more flexible
* It starts fitting **noise and random patterns**
* Tresuction to **artificial red#uction in error**

---

## 🔍 Why All Features Get Selected

During forward selection:

* Each new feature slightly reduces training error
* Even irrelevant features like:

  * `random_id`
  * `noise_feature`

appear to “improve” the model

### Result:

```
Final selected features: ['size', 'rooms', 'age#', 'random_id', 'noise_feature']
```

---

## 🧠 Concept: Overfitting

This behavior is an example of **overfitting**:

* The model learns patterns specific to the training data
* It does not generalize to new/unseen data
* Noi#se is incorrectly treated as useful information

---

## ⚠️ Limitation of This Approach

* Forward selection becomes unreliable
* It cannot distingu#ish between:

  * meaningful features
  * random noise

---

## ✅ What Should Be Done Instead

To make feature selection meaningful, evaluation must be done on **unseen data**:

### Recommend#ed approaches:

* Train-Test Split
* Cross-Validation (preferred)

---

## 🔄 Planned Improvement (Next Step)

In the next implementation:

* Replace training error with validation error
* Use:

  * `train_test_split()` OR
  * `cross_val_score()`

This will ensure:
#
* Only features that **generalize well** are selected
* Noise features are excluded

---

## 🔥 Key Takeaway

> Feature selection is only as good as the evaluation method.
> Using training error leads to selecting all features, including noise.
